In [ ]:
"""# Tweet Hate-Speech and Offensive-Language Classification

## Problem statement

Online platforms receive large volumes of user-generated text that may contain hate speech, offensive language, or harmless content. This project tests whether classical NLP and machine-learning methods can classify tweets into these three categories and assist human review, while making errors, imbalance, and responsible-use limits visible.

## Project objective

This project builds an interpretable NLP baseline for classifying tweets into three categories:

- Hate speech (0)
- Offensive language (1)
- Neither (2)

The notebook compares Bag-of-Words and TF-IDF representations with linear classifiers, then evaluates performance using class-aware metrics, a confusion matrix, and representative prediction errors. Because the labels describe sensitive content, the model is presented as decision support for human review rather than an autonomous moderation system.
"""

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "labeled_data.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "tweets_cleaned.csv"
LABEL_NAMES = {0: "Hate speech", 1: "Offensive language", 2: "Neither"}

sns.set_theme(style="whitegrid", palette="colorblind")
print("Imports ready")

In [ ]:
data = pd.read_csv(DATA_PATH)
data = data[["tweet", "class"]].dropna().copy()
data["class"] = data["class"].astype(int)

print(f"Rows: {len(data):,}")
display(data.head())
display(data["class"].map(LABEL_NAMES).value_counts().rename_axis("label").to_frame("count"))

"Donald John Trump was born on June 14, 1946, at Jamaica Hospital in the New York City borough of Queens, the fourth child of Fred Trump and Mary Anne MacLeod Trump.[1] He is of German and Scottish descent.[2] He grew up with his older siblings, Maryanne, Fred Jr., and Elizabeth, and his younger brother, Robert, in a mansion in the Jamaica Estates neighborhood of Queens.[3] Fred Trump paid his children each about $20,000 a year, equivalent to $265,000 a year in 2024. Trump was a millionaire in inflation-adjusted dollars by age eight.[4][a]\n\nTrump attended the private Kew-Forest School through seventh grade. He was a difficult child and showed an early interest in his father's business. His father enrolled him in New York Military Academy, a private boarding school, to complete secondary school.[5] Trump considered a show business career but instead in 1964 enrolled at Fordham University.[6] Two years later, he transferred to the Wharton School of the University of Pennsylvania, gradu

In [ ]:
"""1. Understand the corpus

The dataset contains labeled tweets and supporting annotator-count columns. The class column is the prediction target. We keep the original tweet text visible during exploration so that preprocessing choices remain explainable.
"""

In [ ]:
label_counts = data["class"].map(LABEL_NAMES).value_counts().reindex(LABEL_NAMES.values())
ax = label_counts.plot(kind="bar", color=["#d95f02", "#1b9e77", "#7570b3"], figsize=(8, 4))
ax.set_title("Class distribution")
ax.set_xlabel("")
ax.set_ylabel("Tweets")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Duplicate tweets:", data["tweet"].duplicated().sum())
print("Missing values:\n", data.isna().sum())

"""2. Prepare social-media text

The cleaner removes URLs, mentions, HTML entities, and repeated whitespace while preserving words and hashtags as useful signals. It intentionally avoids aggressive stemming so the model remains easier to inspect and explain.
"""

In [ ]:
import html
import re


def clean_tweet(text):
    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"\brt\b", " RETWEET ", text, flags=re.IGNORECASE)
    text = re.sub(r"[^\w#'!?\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()


data["clean_tweet"] = data["tweet"].map(clean_tweet)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
data[["tweet", "clean_tweet", "class"]].to_csv(PROCESSED_PATH, index=False)
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_tweet"],
    data["class"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=data["class"],
)
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Saved processed data to: {PROCESSED_PATH}")
print("Example cleaned tweet:")
print(X_train.iloc[0])

## 3. Compare BoW and TF-IDF

Both experiments use the same linear classifiers and split. This makes the representation comparison meaningful rather than mixing changes in preprocessing, model family, and evaluation at the same time.

In [ ]:
models = {
    "BoW + Logistic Regression": Pipeline([
        ("features", CountVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000)),
        ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "TF-IDF + Logistic Regression": Pipeline([
        ("features", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "TF-IDF + Linear SVM": Pipeline([
        ("features", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)),
        ("classifier", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results[name] = {
        "model": model,
        "predictions": predictions,
        "macro_f1": f1_score(y_test, predictions, average="macro"),
    }
    print(f"{name}: macro F1 = {results[name]['macro_f1']:.3f}")

In [ ]:
"""4. Evaluate the best model

Macro F1 gives each class equal importance, which is more informative than accuracy when the labels are not evenly distributed. The confusion matrix shows whether the model struggles most with hate speech versus offensive language.
"""

In [ ]:
best_name = max(results, key=lambda name: results[name]["macro_f1"])
best_model = results[best_name]["model"]
best_predictions = results[best_name]["predictions"]

print(f"Selected model: {best_name}\n")
print(classification_report(
    y_test,
    best_predictions,
    labels=list(LABEL_NAMES),
    target_names=list(LABEL_NAMES.values()),
    digits=3,
    zero_division=0,
))

matrix = confusion_matrix(y_test, best_predictions, labels=list(LABEL_NAMES))
plt.figure(figsize=(7, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="YlOrBr",
    xticklabels=LABEL_NAMES.values(),
    yticklabels=LABEL_NAMES.values(),
)
plt.title(f"Confusion matrix: {best_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

## 5. Inspect mistakes and test new text

Error analysis makes the result more credible than a single score. The examples below expose disagreements for review and provide a small interface for trying unseen tweets.

In [ ]:
review = pd.DataFrame({
    "tweet": data.loc[X_test.index, "tweet"],
    "actual": y_test.map(LABEL_NAMES),
    "predicted": pd.Series(best_predictions, index=y_test.index).map(LABEL_NAMES),
})
errors = review[review["actual"] != review["predicted"]]
display(errors.head(10))
print(f"Misclassified test tweets: {len(errors):,} of {len(y_test):,}")


def predict_tweet(text):
    cleaned = clean_tweet(text)
    prediction = int(best_model.predict([cleaned])[0])
    return LABEL_NAMES[prediction]

examples = [
    "I appreciate your help with this project.",
    "That comment is hateful and targets an entire group.",
]
for example in examples:
    print(f"{predict_tweet(example):<20} | {example}")